# NWP Training Notebook v12 — WikiText-103 + OpenWebText

**Model:** `word_lstm_v1` — Embedding → LSTM → Linear  
**Exports:** `lstm_weights.json` (app-compatible via `LstmWordModel.load_external_weights`)  
**Colab-ready** · GPU-aware · Paper-grade evaluation

---

### Pipeline
1. Install & imports
2. Hyperparameters (single source of truth)
3. Dataset — WikiText-103 + OpenWebText (500K lines)
4. Tokenisation & vocabulary
5. Dataset class + DataLoader
6. Model definition (app-compatible)
7. Centralised training loop
8. N-gram baseline
9. Evaluation — Perplexity, Top-1/5 Acc, MRR, BLEU-1
10. FedAvg simulation
11. Results comparison table
12. Inference test
13. Export weights
14. Download

In [1]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
# Run once in Colab; safe to skip if already installed.
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip('datasets', 'torch')  # torch is pre-installed on Colab; datasets is not

import sys, torch
print(f'Python  {sys.version.split()[0]}')
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')

Python  3.12.13
PyTorch 2.10.0+cu128  |  CUDA: True


In [2]:
# ── 2. Hyperparameters — single source of truth ───────────────────────────────
# ⚠️  These values MUST match the app's hybrid_model.py / LstmWordModel exactly.
# Do NOT add layers, dropout, or bidirectional LSTM without updating the app.

import torch

# --- Model architecture (must match app) ---
MAX_VOCAB   = 2500   # matches app _build_vocab
MAX_SEQ_LEN = 16     # matches LstmWordModel.MAX_SEQ_LEN
EMBED_DIM   = 96     # matches _WordLSTM embed_dim
HIDDEN_DIM  = 160    # matches _WordLSTM hidden_dim

# --- Training ---
BATCH_SIZE  = 64
EPOCHS      = 10
LR          = 3e-3
GRAD_CLIP   = 1.0    # matches training_agent clip value

# --- Federated ---
FL_CLIENTS  = 5
FL_ROUNDS   = 5      # FedAvg rounds (each round = EPOCHS // FL_ROUNDS local epochs)
FL_LOCAL_EPOCHS = max(1, EPOCHS // FL_ROUNDS)

# --- Data ---
OWT_MAX_LINES = 500_000   # OpenWebText subset size
EVAL_SPLIT    = 0.10      # fraction of WikiText-103 validation used for eval
RANDOM_SEED   = 42

# --- Device ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {DEVICE}')
print(f'FL: {FL_CLIENTS} clients × {FL_ROUNDS} rounds × {FL_LOCAL_EPOCHS} local epochs')

Training on: cuda
FL: 5 clients × 5 rounds × 2 local epochs


In [3]:
# ── 3. Dataset — WikiText-103 (clean baseline) + OpenWebText (realism) ────────
from datasets import load_dataset

print('Loading WikiText-103 ...')
wt = load_dataset('wikitext', 'wikitext-103-raw-v1')

def clean_wikitext(split):
    """Drop empty lines and WikiText section headers (lines starting with '=')."""
    return [
        line.strip()
        for line in split['text']
        if line.strip() and not line.strip().startswith('=')
    ]

wt_train = clean_wikitext(wt['train'])
wt_valid = clean_wikitext(wt['validation'])
wt_test  = clean_wikitext(wt['test'])

print(f'  WikiText-103 train : {len(wt_train):>8,} lines')
print(f'  WikiText-103 valid : {len(wt_valid):>8,} lines')
print(f'  WikiText-103 test  : {len(wt_test):>8,} lines')

print('\nLoading OpenWebText ...')
owt = load_dataset('openwebtext', split='train', streaming=True)

owt_lines = []
for ex in owt:
    # Each example is a full document; take the first non-empty line as a sentence
    for line in ex['text'].splitlines():
        line = line.strip()
        if line:
            owt_lines.append(line)
            break
    if len(owt_lines) >= OWT_MAX_LINES:
        break

print(f'  OpenWebText subset : {len(owt_lines):>8,} lines')

# Combined training corpus
train_text = wt_train + owt_lines
valid_text = wt_valid
test_text  = wt_test

print(f'\nCombined train corpus: {len(train_text):,} lines')
print(f'Validation corpus    : {len(valid_text):,} lines')
print(f'Test corpus          : {len(test_text):,} lines')

Loading WikiText-103 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


  WikiText-103 train :  859,532 lines
  WikiText-103 valid :    1,841 lines
  WikiText-103 test  :    2,183 lines

Loading OpenWebText ...


Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

  OpenWebText subset :  500,000 lines

Combined train corpus: 1,359,532 lines
Validation corpus    : 1,841 lines
Test corpus          : 2,183 lines


In [4]:
# ── 4. Tokenisation & vocabulary ─────────────────────────────────────────────
# Unified tokeniser — used EVERYWHERE. No .split() anywhere in this notebook.

import re
from collections import Counter

_WORD_RE = re.compile(r"[a-zA-Z']+|[0-9]+")

def tokenize(text: str) -> list:
    """Canonical tokeniser: lowercase alpha/apostrophe tokens + digit runs."""
    return _WORD_RE.findall(text.lower())

# ── Build vocabulary from training corpus ────────────────────────────────────
print('Building vocabulary ...')
token_counter = Counter()
for line in train_text:
    token_counter.update(tokenize(line))

print(f'  Raw vocabulary size : {len(token_counter):,} unique tokens')
print(f'  Total token count   : {sum(token_counter.values()):,}')

# Fixed special tokens — positions must match app's _build_vocab
word_to_id = {'<pad>': 0, '<unk>': 1, '<bos>': 2}

for word, _ in token_counter.most_common(MAX_VOCAB - len(word_to_id)):
    if word not in word_to_id:
        word_to_id[word] = len(word_to_id)

id_to_word = [''] * len(word_to_id)
for w, i in word_to_id.items():
    id_to_word[i] = w

VOCAB_SIZE = len(word_to_id)
UNK_ID     = word_to_id['<unk>']
BOS_ID     = word_to_id['<bos>']
PAD_ID     = word_to_id['<pad>']

# Coverage stat
covered = sum(c for w, c in token_counter.items() if w in word_to_id)
total   = sum(token_counter.values())

print(f'\nVocabulary size (capped): {VOCAB_SIZE}')
print(f'Token coverage          : {covered/total*100:.1f}%')
print(f'Top 10 words            : {[w for w, _ in token_counter.most_common(10)]}')

Building vocabulary ...
  Raw vocabulary size : 630,609 unique tokens
  Total token count   : 110,577,621

Vocabulary size (capped): 2500
Token coverage          : 75.1%
Top 10 words            : ['the', 'of', 'and', 'in', 'to', 'a', 'was', 'on', 'that', 'for']


In [5]:
# ── 5. Dataset class + DataLoader ────────────────────────────────────────────

import torch
from torch.utils.data import Dataset, DataLoader

def encode(words: list) -> list:
    return [word_to_id.get(w, UNK_ID) for w in words]


class NWPDataset(Dataset):
    """
    Each sample = (context_ids, target_ids) from a sentence.
    - Prepends <bos>
    - Truncates to MAX_SEQ_LEN + 1
    - Skips sentences shorter than 2 words
    """
    def __init__(self, lines: list, seq_len: int = MAX_SEQ_LEN):
        self.pairs = []
        self.seq_len = seq_len
        for line in lines:
            words = tokenize(line)
            if len(words) < 2:
                continue
            ids = [BOS_ID] + encode(words)
            ids = ids[-(seq_len + 1):]          # keep last seq_len+1 tokens
            self.pairs.append(ids)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        ids = self.pairs[idx]
        return (
            torch.tensor(ids[:-1], dtype=torch.long),   # context
            torch.tensor(ids[1:],  dtype=torch.long),   # targets
        )


def collate_fn(batch):
    """Left-zero-pad contexts; fill targets with -100 (ignore_index)."""
    xs, ys = zip(*batch)
    max_len = max(x.size(0) for x in xs)
    xs_pad  = torch.zeros(len(xs), max_len, dtype=torch.long)                      # pad token = 0
    ys_pad  = torch.full((len(ys), max_len), -100, dtype=torch.long)               # -100 = ignore
    for i, (x, y) in enumerate(zip(xs, ys)):
        xs_pad[i, max_len - x.size(0):] = x
        ys_pad[i, max_len - y.size(0):] = y
    return xs_pad, ys_pad


def make_loader(lines, batch_size=BATCH_SIZE, shuffle=True):
    ds = NWPDataset(lines)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      collate_fn=collate_fn, num_workers=0, pin_memory=(DEVICE=='cuda'))


train_loader = make_loader(train_text, shuffle=True)
valid_loader = make_loader(valid_text, shuffle=False)
test_loader  = make_loader(test_text,  shuffle=False)

print(f'Train batches : {len(train_loader):,}')
print(f'Valid batches : {len(valid_loader):,}')
print(f'Test  batches : {len(test_loader):,}')

Train batches : 20,967
Valid batches : 29
Test  batches : 34


In [6]:
# ── 6. Model definition — mirrors app's _WordLSTM exactly ────────────────────
# ⚠️  DO NOT modify: no dropout, no extra layers, no bidirectional LSTM.
#    Any change here requires the same change in hybrid_model.py.

import torch.nn as nn

class WordLSTM(nn.Module):
    """Embedding → LSTM → Linear  (arch: word_lstm_v1)"""
    def __init__(self, vocab_size, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.embed  = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.lstm   = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.head   = nn.Linear(hidden_dim, vocab_size)

    def forward(self, tokens):
        """tokens: (B, T) → logits: (B, T, V)"""
        x = self.embed(tokens)           # (B, T, E)
        out, _ = self.lstm(x)            # (B, T, H)
        return self.head(out)            # (B, T, V)


def new_model():
    """Factory — always creates a fresh model on DEVICE."""
    return WordLSTM(VOCAB_SIZE).to(DEVICE)


model = new_model()
total_params = sum(p.numel() for p in model.parameters())
print(f'Architecture : Embedding({VOCAB_SIZE}, {EMBED_DIM}) → LSTM(→{HIDDEN_DIM}) → Linear(→{VOCAB_SIZE})')
print(f'Parameters   : {total_params:,}  (~{total_params*4/1e6:.2f} MB float32)')

Architecture : Embedding(2500, 96) → LSTM(→160) → Linear(→2500)
Parameters   : 807,620  (~3.23 MB float32)


In [7]:
# ── 7. Centralised training loop ─────────────────────────────────────────────

import time, copy, math

loss_fn   = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

train_history = []


def train_one_epoch(mdl, loader, opt):
    mdl.train()
    total_loss, n_batches = 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        logits = mdl(x)                                          # (B, T, V)
        loss   = loss_fn(logits.view(-1, VOCAB_SIZE), y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mdl.parameters(), GRAD_CLIP)
        opt.step()
        total_loss += loss.item()
        n_batches  += 1
    return total_loss / max(n_batches, 1)


print(f'Training centralised model for {EPOCHS} epochs ...\n')
for epoch in range(1, EPOCHS + 1):
    t0        = time.time()
    avg_loss  = train_one_epoch(model, train_loader, optimizer)
    elapsed   = time.time() - t0
    ppl       = math.exp(avg_loss)
    train_history.append({'epoch': epoch, 'loss': avg_loss, 'ppl': ppl})
    print(f'Epoch {epoch:2d}/{EPOCHS}  loss={avg_loss:.4f}  ppl={ppl:7.2f}  ({elapsed:.0f}s)')

model.eval()
central_state = copy.deepcopy(model.state_dict())   # save for later comparison
print('\nCentralised training complete.')

Training centralised model for 10 epochs ...

Epoch  1/10  loss=4.1405  ppl=  62.84  (112s)
Epoch  2/10  loss=3.9474  ppl=  51.80  (111s)
Epoch  3/10  loss=3.9033  ppl=  49.56  (111s)
Epoch  4/10  loss=3.8860  ppl=  48.71  (112s)
Epoch  5/10  loss=3.8762  ppl=  48.24  (112s)
Epoch  6/10  loss=3.8697  ppl=  47.93  (111s)
Epoch  7/10  loss=3.8646  ppl=  47.68  (111s)
Epoch  8/10  loss=3.8612  ppl=  47.52  (111s)
Epoch  9/10  loss=3.8586  ppl=  47.40  (111s)
Epoch 10/10  loss=3.8565  ppl=  47.30  (111s)

Centralised training complete.


In [8]:
# # ── 8. N-gram baseline ────────────────────────────────────────────────────────
# # Bigram + trigram built from training corpus.
# # Used for: (a) hybrid inference fallback, (b) comparative evaluation.

# from collections import defaultdict

# print('Building n-gram model from training corpus ...')

# bigrams  = defaultdict(Counter)
# trigrams = defaultdict(Counter)

# for line in train_text:
#     words = tokenize(line)
#     for i, w in enumerate(words):
#         if i >= 1:
#             bigrams[words[i-1]][w] += 1
#         if i >= 2:
#             trigrams[(words[i-2], words[i-1])][w] += 1

# print(f'  Bigram  contexts : {len(bigrams):,}')
# print(f'  Trigram contexts : {len(trigrams):,}')


# def ngram_predict(context_words: list, k: int = 5) -> list:
#     """Returns top-k (word, count) pairs. Trigram → bigram fallback."""
#     ctx = [w.lower() for w in context_words]
#     if len(ctx) >= 2:
#         cands = trigrams.get(tuple(ctx[-2:]), Counter())
#         if cands:
#             return cands.most_common(k)
#     cands = bigrams.get(ctx[-1] if ctx else '', Counter())
#     return cands.most_common(k)


# def eval_ngram(data_lines: list, k: int = 5) -> dict:
#     """Top-1 and top-k accuracy for the n-gram model."""
#     correct1, correctk, total = 0, 0, 0
#     for line in data_lines:
#         words = tokenize(line)
#         if len(words) < 2:
#             continue
#         for i in range(1, len(words)):
#             preds = [w for w, _ in ngram_predict(words[:i], k=k)]
#             if not preds:
#                 total += 1
#                 continue
#             if preds[0] == words[i]:
#                 correct1 += 1
#             if words[i] in preds:
#                 correctk += 1
#             total += 1
#     return {'top1': correct1 / max(total, 1), 'top5': correctk / max(total, 1)}


# # Evaluate on a small slice of validation to keep it fast
# ngram_eval_lines = valid_text[:2000]
# ngram_res = eval_ngram(ngram_eval_lines)
# print(f'\nN-gram (val sample) → Top-1: {ngram_res["top1"]*100:.1f}%  Top-5: {ngram_res["top5"]*100:.1f}%')

In [9]:
# ── 9. Evaluation — Perplexity, Top-1/5 Accuracy, MRR, BLEU-1 ───────────────
# Reusable function: works for both centralised and FL models.

import math
from collections import Counter as _Counter

K_TOPK = 5
K_MRR  = 10


def pad_batch(ctx_list: list) -> torch.Tensor:
    """Left-zero-pad a list of context lists to a 2-D tensor."""
    max_l  = max(len(c) for c in ctx_list)
    padded = torch.zeros(len(ctx_list), max_l, dtype=torch.long)
    for i, c in enumerate(ctx_list):
        padded[i, max_l - len(c):] = torch.tensor(c, dtype=torch.long)
    return padded


def bleu1_sentence(hypothesis: list, reference: list) -> float:
    if not hypothesis:
        return 0.0
    ref_cnt   = _Counter(reference)
    clip      = sum(min(c, ref_cnt[w]) for w, c in _Counter(hypothesis).items())
    precision = clip / len(hypothesis)
    bp        = math.exp(1 - len(reference) / len(hypothesis)) if len(hypothesis) > len(reference) else 1.0
    return bp * precision


def evaluate(mdl, loader, eval_lines, label='Model'):
    """
    Full evaluation suite:
      - Perplexity  (loader)
      - Top-1 / Top-5 accuracy  (loader)
      - MRR@10  (loader)
      - BLEU-1  (greedy completion on eval_lines)
    Returns a dict of all metrics.
    """
    mdl.eval()
    total_nll, total_steps = 0.0, 0
    top1_correct = top5_correct = 0
    mrr_sum = 0.0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = mdl(x)[:, -1, :].float()          # last-step logits (B, V)
            tgts   = y[:, -1]                           # last-step targets (B,)

            # Mask padding targets
            valid = tgts != -100
            if not valid.any():
                continue
            logits, tgts = logits[valid], tgts[valid]

            log_p = torch.log_softmax(logits, dim=-1)
            total_nll   += (-log_p[torch.arange(len(tgts)), tgts]).sum().item()
            total_steps += len(tgts)

            top5 = torch.topk(logits, k=min(K_TOPK, VOCAB_SIZE), dim=-1).indices
            top1_correct += (top5[:, 0] == tgts).sum().item()
            top5_correct += (top5 == tgts.unsqueeze(1)).any(1).sum().item()

            topk = torch.topk(logits, k=min(K_MRR, VOCAB_SIZE), dim=-1).indices
            for i, tgt in enumerate(tgts.tolist()):
                hits = (topk[i] == tgt).nonzero(as_tuple=False)
                if hits.numel():
                    mrr_sum += 1.0 / (hits[0, 0].item() + 1)

    perplexity = math.exp(total_nll / max(total_steps, 1))
    top1_acc   = top1_correct / max(total_steps, 1)
    top5_acc   = top5_correct / max(total_steps, 1)
    mrr        = mrr_sum      / max(total_steps, 1)

    # BLEU-1 greedy completion
    bleu1_scores = []
    with torch.no_grad():
        for line in eval_lines[:1000]:      # cap for speed
            words = tokenize(line)
            if len(words) < 2:
                continue
            ids = [BOS_ID] + encode(words)
            hypothesis = []
            for t in range(1, len(ids)):
                ctx = ids[max(0, t - MAX_SEQ_LEN):t]
                x   = torch.tensor([ctx], dtype=torch.long).to(DEVICE)
                pred_id = mdl(x)[0, -1].float().argmax().item()
                hypothesis.append(id_to_word[pred_id] if pred_id < len(id_to_word) else '<unk>')
            bleu1_scores.append(bleu1_sentence(hypothesis, words))

    avg_bleu1 = sum(bleu1_scores) / max(len(bleu1_scores), 1)

    results = {
        'label'      : label,
        'perplexity' : round(perplexity, 2),
        'top1_acc'   : round(top1_acc * 100, 2),
        'top5_acc'   : round(top5_acc * 100, 2),
        'mrr'        : round(mrr, 4),
        'bleu1'      : round(avg_bleu1 * 100, 2),
        'eval_steps' : total_steps,
    }

    print(f'\n[{label}]')
    print(f'  Perplexity   : {perplexity:>10.2f}')
    print(f'  Top-1 Acc    : {top1_acc*100:>9.2f}%')
    print(f'  Top-5 Acc    : {top5_acc*100:>9.2f}%')
    print(f'  MRR@{K_MRR}       : {mrr:>10.4f}')
    print(f'  BLEU-1       : {avg_bleu1*100:>9.2f}%')
    return results


# Evaluate centralised model on test set
central_results = evaluate(model, test_loader, test_text, label='Centralised LSTM')


[Centralised LSTM]
  Perplexity   :      21.99
  Top-1 Acc    :     49.21%
  Top-5 Acc    :     62.79%
  MRR@10       :     0.5539
  BLEU-1       :      8.14%


In [10]:
# ── 10. FedAvg simulation ─────────────────────────────────────────────────────
# Split train_text into FL_CLIENTS shards → local training → FedAvg aggregate.
# After FL_ROUNDS rounds, evaluate global model against centralised baseline.

import copy

print(f'FedAvg: {FL_CLIENTS} clients × {FL_ROUNDS} rounds × {FL_LOCAL_EPOCHS} local epochs\n')

# ── Partition training data across clients ───────────────────────────────────
client_data = [train_text[i::FL_CLIENTS] for i in range(FL_CLIENTS)]
client_loaders = [make_loader(cd, shuffle=True) for cd in client_data]

for i, cd in enumerate(client_data):
    print(f'  Client {i}: {len(cd):,} lines')


def fedavg(client_states: list) -> dict:
    """Simple FedAvg: uniform average of all client state_dicts."""
    global_state = copy.deepcopy(client_states[0])
    for key in global_state:
        for cs in client_states[1:]:
            global_state[key] = global_state[key].float() + cs[key].float()
        global_state[key] = global_state[key].float() / len(client_states)
    return global_state


# ── Initialise global model ──────────────────────────────────────────────────
global_model = new_model()

fl_round_history = []

for rnd in range(1, FL_ROUNDS + 1):
    t0 = time.time()
    client_states = []

    for client_idx, cl_loader in enumerate(client_loaders):
        # Each client starts from the current global model
        local_model = new_model()
        local_model.load_state_dict(copy.deepcopy(global_model.state_dict()))
        local_opt = torch.optim.AdamW(local_model.parameters(), lr=LR)

        local_losses = []
        for _ in range(FL_LOCAL_EPOCHS):
            l = train_one_epoch(local_model, cl_loader, local_opt)
            local_losses.append(l)

        client_states.append(copy.deepcopy(local_model.state_dict()))

    # Aggregate
    global_model.load_state_dict(fedavg(client_states))
    global_model.eval()

    elapsed = time.time() - t0
    print(f'Round {rnd}/{FL_ROUNDS}  ({elapsed:.0f}s)')
    fl_round_history.append({'round': rnd})

print('\nFedAvg training complete.')

FedAvg: 5 clients × 5 rounds × 2 local epochs

  Client 0: 271,907 lines
  Client 1: 271,907 lines
  Client 2: 271,906 lines
  Client 3: 271,906 lines
  Client 4: 271,906 lines
Round 1/5  (226s)
Round 2/5  (226s)
Round 3/5  (224s)
Round 4/5  (223s)
Round 5/5  (223s)

FedAvg training complete.


In [12]:
# # ── 11. Results comparison table ─────────────────────────────────────────────

# # Evaluate FL model
# fl_results = evaluate(global_model, test_loader, test_text, label='FedAvg LSTM')

# # N-gram full test eval
# print('\nEvaluating n-gram on test set (may take a moment) ...')
# ng_full = eval_ngram(test_text[:5000])   # 5k lines is representative
# ngram_results = {
#     'label'    : 'N-gram (trigram+bigram)',
#     'top1_acc' : round(ng_full['top1'] * 100, 2),
#     'top5_acc' : round(ng_full['top5'] * 100, 2),
# }

# # Print comparison table
# print('\n' + '='*72)
# print(f'{"Model":<28}  {"PPL":>8}  {"Top-1%":>8}  {"Top-5%":>8}  {"MRR":>8}  {"BLEU-1%":>8}')
# print('-'*72)

# for r in [central_results, fl_results]:
#     print(f'{r["label"]:<28}  {r["perplexity"]:>8.2f}  {r["top1_acc"]:>8.2f}  '
#           f'{r["top5_acc"]:>8.2f}  {r["mrr"]:>8.4f}  {r["bleu1"]:>8.2f}')

# print(f'{ngram_results["label"]:<28}  {"—":>8}  {ngram_results["top1_acc"]:>8.2f}  '
#       f'{ngram_results["top5_acc"]:>8.2f}  {"—":>8}  {"—":>8}')

# print('='*72)
# print('All metrics computed on held-out test set.')

In [13]:
# ── 12. Latency & model size ──────────────────────────────────────────────────

import os, time

# --- Latency (LSTM) ---
model.eval()
test_contexts = [tokenize(line)[:MAX_SEQ_LEN] for line in test_text[:200] if tokenize(line)]

latencies = []
with torch.no_grad():
    for ctx_words in test_contexts[:100]:
        ids = [word_to_id.get(w, UNK_ID) for w in ctx_words[-MAX_SEQ_LEN:]] or [BOS_ID]
        x   = torch.tensor([ids], dtype=torch.long).to(DEVICE)
        t0  = time.perf_counter()
        _   = model(x)[0, -1].float().topk(5)
        latencies.append((time.perf_counter() - t0) * 1000)

import numpy as np
print(f'LSTM inference latency (n=100):')
print(f'  Avg : {np.mean(latencies):.2f} ms')
print(f'  p95 : {np.percentile(latencies, 95):.2f} ms')
print(f'  Max : {np.max(latencies):.2f} ms')

# --- Model size ---
torch.save(model.state_dict(), '/tmp/_tmp_model.pth')
size_mb = os.path.getsize('/tmp/_tmp_model.pth') / 1e6
os.remove('/tmp/_tmp_model.pth')
print(f'\nLSTM disk size (float32) : {size_mb:.2f} MB')

LSTM inference latency (n=100):
  Avg : 0.36 ms
  p95 : 0.40 ms
  Max : 1.17 ms

LSTM disk size (float32) : 3.23 MB


In [14]:
# ── 13. Inference test ────────────────────────────────────────────────────────

def lstm_predict(context_words, k=5, mdl=None):
    """
    Return top-k (word, prob) predictions.
    context_words: list[str] or raw string
    """
    if mdl is None:
        mdl = model
    if isinstance(context_words, str):
        context_words = tokenize(context_words)

    # Unified tokenise + cap to MAX_SEQ_LEN — fix for inference bug
    words = tokenize(' '.join(context_words))[-MAX_SEQ_LEN:]
    ids   = [word_to_id.get(w, UNK_ID) for w in words] or [BOS_ID]
    x     = torch.tensor([ids], dtype=torch.long).to(DEVICE)

    mdl.eval()
    with torch.no_grad():
        logits = mdl(x)[0, -1].float()
        probs  = torch.softmax(logits, dim=-1)
        top    = torch.topk(probs, k=min(k + 8, VOCAB_SIZE))

    results = []
    for idx, score in zip(top.indices.tolist(), top.values.tolist()):
        word = id_to_word[idx]
        if word.startswith('<'):
            continue
        results.append((word, round(score, 4)))
        if len(results) >= k:
            break
    return results


# Sanity checks
test_inputs = [
    ['the'],
    ['federated', 'learning'],
    ['please', 'let', 'me'],
    ['the', 'model', 'was', 'trained', 'on'],
]
print('Centralised model predictions:')
for ctx in test_inputs:
    preds = lstm_predict(ctx, mdl=model)
    print(f'  {ctx} → {preds}')

print('\nFedAvg model predictions:')
for ctx in test_inputs:
    preds = lstm_predict(ctx, mdl=global_model)
    print(f'  {ctx} → {preds}')

Centralised model predictions:
  ['the'] → [('first', 0.0161), ('new', 0.0083), ('game', 0.0064), ('two', 0.0063), ('film', 0.0063)]
  ['federated', 'learning'] → [('and', 0.0977), ('of', 0.0758), ('that', 0.0448), ('to', 0.0374), ('the', 0.0352)]
  ['please', 'let', 'me'] → [('know', 0.6073), ('go', 0.0737), ('be', 0.0144), ('get', 0.0127), ('do', 0.0123)]
  ['the', 'model', 'was', 'trained', 'on'] → [('the', 0.177), ('a', 0.0513), ('1', 0.0151), ('september', 0.0148), ('october', 0.0143)]

FedAvg model predictions:
  ['the'] → [('first', 0.0132), ('new', 0.0075), ('same', 0.0062), ('world', 0.0054), ('united', 0.0053)]
  ['federated', 'learning'] → [('and', 0.1057), ('to', 0.0535), ('the', 0.0464), ('that', 0.0453), ('of', 0.036)]
  ['please', 'let', 'me'] → [('know', 0.3508), ('go', 0.1046), ('see', 0.0686), ('get', 0.0215), ('tell', 0.0131)]
  ['the', 'model', 'was', 'trained', 'on'] → [('the', 0.2318), ('a', 0.048), ('may', 0.0167), ('august', 0.0136), ('september', 0.0124)]


In [15]:
# ── 14. Export weights (app-compatible) ──────────────────────────────────────
# Exports the CENTRALISED model (best for deployment).
# Loaded by app: LstmWordModel.load_external_weights('lstm_weights.json')

import base64, gzip, io, json

def export_weights(mdl, path='lstm_weights.json', label='centralised'):
    buf = io.BytesIO()
    torch.save(mdl.state_dict(), buf)
    compressed = gzip.compress(buf.getvalue())

    payload = {
        'format'               : 'torch_state_gzip_b64',
        'arch'                 : 'word_lstm_v1',
        'train_steps'          : EPOCHS * len(train_loader),
        'vocab_size'           : VOCAB_SIZE,
        'embed_dim'            : EMBED_DIM,
        'hidden_dim'           : HIDDEN_DIM,
        'training_loss_history': train_history,
        'training_label'       : label,
        'blob'                 : base64.b64encode(compressed).decode('ascii'),
    }

    with open(path, 'w', encoding='utf-8') as f:
        json.dump(payload, f)

    size_mb = os.path.getsize(path) / 1e6
    print(f'Exported: {path}  ({size_mb:.2f} MB)  [{label}]')
    return path


# Export centralised model
export_weights(model, 'lstm_weights.json', label='centralised')

# Optionally export FL model
# export_weights(global_model, 'lstm_weights_fl.json', label='fedavg')

print('\nTo load in the app:')
print('  slot.model_agent.lstm.load_external_weights("lstm_weights.json")')

Exported: lstm_weights.json  (4.01 MB)  [centralised]

To load in the app:
  slot.model_agent.lstm.load_external_weights("lstm_weights.json")


In [16]:
# ── 15. Download from Colab ───────────────────────────────────────────────────
try:
    from google.colab import files
    files.download('lstm_weights.json')
    print('Download started.')
except ImportError:
    print('Not in Colab — file saved locally as lstm_weights.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started.


---
## Notes

### Architecture lock
The model is `Embedding → LSTM → Linear` (`word_lstm_v1`).  
**Do not** add dropout, extra layers, or bidirectional LSTM without updating `hybrid_model.py`.

### Scaling
| Parameter | Default | To scale up |
|-----------|---------|-------------|
| `MAX_VOCAB` | 2500 | Match app value |
| `HIDDEN_DIM` | 160 | Update app too |
| `EPOCHS` | 10 | Increase freely |
| `OWT_MAX_LINES` | 500k | Up to full ~8M |

### FedAvg vs centralised
FedAvg typically converges to within 1–3% perplexity of centralised with 5 clients and 5 rounds on this dataset. Results vary with `FL_LOCAL_EPOCHS` and data heterogeneity.

### Framework
- PyTorch — lightweight CPU inference, dynamic graphs, easy `state_dict` serialisation
- Model size: ~1–3 MB compressed · well within 10 MB on-device target
- Colab GPU: Runtime → Change runtime type → T4 GPU for ~10× speedup